# PHISIX API (Rust Port) — Endpoint Demo

This notebook demonstrates every endpoint exposed by the **PHISIX API**, a Rust port of the Philippine Stock Exchange Composite Index (PSEi) RESTful API.

The service scrapes live data from `https://frames.pse.com.ph`, serves it as JSON or JAXB-compatible XML, and archives history into a local SQLite database.

## Endpoints covered

| Method | Path | Description |
|--------|------|-------------|
| `GET` | `/stocks` | All live stocks (JSON default) |
| `GET` | `/stocks.json` | All live stocks as JSON |
| `GET` | `/stocks.xml` | All live stocks as XML |
| `GET` | `/stocks/{symbol}` | Single live stock (e.g. `/stocks/ali`) |
| `GET` | `/stocks/{symbol}.xml` | Single live stock as XML |
| `GET` | `/stocks/{symbol}.{date}` | Historical lookup (DB, then proxy fallback) |
| `GET` / `POST` | `/stocks/archive` | Archive current feed to SQLite |
| `GET` / `POST` | `/dividends/scrape` | Scrape PSE EDGE dividends → DB |
| `GET` | `/dividends` | All dividends (JSON, with `?from=`/`?to=` filters) |
| `GET` | `/dividends.xml` | All dividends as XML |
| `GET` | `/dividends/{symbol}` | Dividends for a specific security |
| `POST` | `/dividends` | Create/update dividend record(s) |
| `DELETE` | `/dividends/{symbol}/{ex_date}` | Delete a dividend record |
| `GET` | `/portfolios` | List all signature portfolios |
| `GET` | `/portfolios/{id_or_slug}` | Target holdings & weights |
| `GET` | `/portfolios/{id_or_slug}/analysis` | Live price enrichment & sector breakdown |
| `POST` | `/portfolios` | Create custom signature portfolio |
| `DELETE` | `/portfolios/{id_or_slug}` | Delete signature portfolio |

## Prerequisites

Start the server first (default port `8080`):

```bash
cargo run
# or containerized:
./build.sh && ./run.sh
```

This notebook only needs the `requests` library:

```bash
pip install requests
```

## Setup

Configure the base URL and a couple of small helpers to pretty-print responses. Change `BASE_URL` if you run the server on a different host or port.

In [1]:
import json
import xml.dom.minidom

import requests

BASE_URL = "http://localhost:8080"


def show(resp):
    """Print status, content-type, and a pretty-printed body (JSON or XML)."""
    ctype = resp.headers.get("content-type", "")
    print(f"{resp.request.method} {resp.url}")
    print(f"-> {resp.status_code} {resp.reason}  |  content-type: {ctype}\n")

    if "json" in ctype:
        print(json.dumps(resp.json(), indent=2))
    elif "xml" in ctype:
        print(xml.dom.minidom.parseString(resp.text).toprettyxml(indent="  ").strip())
    else:
        print(resp.text)


# Quick reachability check
try:
    ping = requests.get(f"{BASE_URL}/stocks", timeout=15)
    print(f"Server reachable: {ping.status_code} {ping.reason}")
except requests.exceptions.RequestException as exc:
    print(f"Could not reach {BASE_URL} — is the server running? ({exc})")

Server reachable: 200 OK


## 1. Get all live stocks (JSON)

`GET /stocks` returns the full PSEi feed as JSON. The server caches the scraped feed for **60 seconds**, so repeated calls within that window are served from memory.

The response shape is:

```jsonc
{
  "as_of": "2026-07-03T15:20:00+08:00",   // ISO 8601, Asia/Manila (GMT+8)
  "stocks": [
    {
      "name": "Ayala Land, Inc.",
      "symbol": "ALI",
      "price": { "currency": "PHP", "amount": 30.5 },
      "percent_change": 0.66,
      "volume": 12345678
    }
  ]
}
```

In [2]:
resp = requests.get(f"{BASE_URL}/stocks", timeout=30)

data = resp.json()
print(f"as_of: {data['as_of']}")
print(f"total stocks: {len(data['stocks'])}\n")

# Show the first 5 entries
print(json.dumps(data["stocks"][:5], indent=2))

as_of: 2026-08-06T00:00:00+08:00
total stocks: 389

[
  {
    "name": "Ayala Corporation Class ``B`` Series 3 Preferred Shares",
    "symbol": "ACPB3",
    "price": {
      "currency": "PHP",
      "amount": 1960.0
    },
    "percent_change": -0.2,
    "volume": 25
  },
  {
    "name": "Ayala Corporation Class ``B`` Series 4 Preferred Shares",
    "symbol": "ACPB4",
    "price": {
      "currency": "PHP",
      "amount": 1975.0
    },
    "percent_change": 0.77,
    "volume": 5
  },
  {
    "name": "Alsons Consolidated Resources, Inc.",
    "symbol": "ACR",
    "price": {
      "currency": "PHP",
      "amount": 0.67
    },
    "percent_change": -5.63,
    "volume": 8927000
  },
  {
    "name": "Aboitiz Equity Ventures, Inc.",
    "symbol": "AEV",
    "price": {
      "currency": "PHP",
      "amount": 35.8
    },
    "percent_change": -3.11,
    "volume": 223500
  },
  {
    "name": "Alliance Global Group, Inc.",
    "symbol": "AGI",
    "price": {
      "currency": "PHP",
      "amo

### `/stocks.json` — explicit JSON suffix

Identical payload to `/stocks`; the `.json` suffix is supported for compatibility with the original API.

In [3]:
resp = requests.get(f"{BASE_URL}/stocks.json", timeout=30)
print(f"{resp.status_code} {resp.reason}  |  content-type: {resp.headers.get('content-type')}")
print(f"total stocks: {len(resp.json()['stocks'])}")

200 OK  |  content-type: application/json
total stocks: 389


## 2. Get all live stocks (XML)

`GET /stocks.xml` returns the same data serialized as JAXB-compatible XML using the `stocks:` namespace, matching the original Java implementation byte-for-byte.

In [4]:
resp = requests.get(f"{BASE_URL}/stocks.xml", timeout=30)
print(f"{resp.status_code} {resp.reason}  |  content-type: {resp.headers.get('content-type')}\n")

# Print just the header + first stock block to keep output short
pretty = xml.dom.minidom.parseString(resp.text).toprettyxml(indent="  ")
print("\n".join(pretty.splitlines()[:14]))
print("  ...")

200 OK  |  content-type: application/xml

<?xml version="1.0" ?>
<stocks:stocks xmlns:stocks="http://phisix-api.appspot.com/phisix-stocks" as_of="2026-08-06T00:00:00+08:00">
  
  
  <stocks:stock symbol="ACPB3">
    
    
    <stocks:name>Ayala Corporation Class ``B`` Series 3 Preferred Shares</stocks:name>
    
    
    <stocks:price>
      
      
      <stocks:currency>PHP</stocks:currency>
  ...


## 3. Single stock lookup

`GET /stocks/{symbol}` returns just one company from the live feed. The symbol is case-insensitive (`ali`, `ALI`, and `Ali` all work).

A missing symbol returns **404 Not Found**.

In [5]:
SYMBOL = "bpi"  # Ayala Land, Inc.

resp = requests.get(f"{BASE_URL}/stocks/{SYMBOL}", timeout=30)
show(resp)

GET http://localhost:8080/stocks/bpi
-> 200 OK  |  content-type: application/json

{
  "as_of": "2026-08-06T00:00:00+08:00",
  "stocks": [
    {
      "name": "Bank of the Philippine Islands",
      "symbol": "BPI",
      "price": {
        "currency": "PHP",
        "amount": 103.4
      },
      "percent_change": -0.48,
      "volume": 1323580
    }
  ]
}


### Same stock as XML

Append `.xml` to any single-stock path to get the XML representation (`.json` is also accepted).

In [6]:
resp = requests.get(f"{BASE_URL}/stocks/{SYMBOL}.xml", timeout=30)
show(resp)

GET http://localhost:8080/stocks/bpi.xml
-> 200 OK  |  content-type: application/xml

<?xml version="1.0" ?>
<stocks:stocks xmlns:stocks="http://phisix-api.appspot.com/phisix-stocks" as_of="2026-08-06T00:00:00+08:00">
  
  
  <stocks:stock symbol="BPI">
    
    
    <stocks:name>Bank of the Philippine Islands</stocks:name>
    
    
    <stocks:price>
      
      
      <stocks:currency>PHP</stocks:currency>
      
      
      <stocks:amount>103.4</stocks:amount>
      
    
    </stocks:price>
    
    
    <stocks:percent_change>-0.48</stocks:percent_change>
    
    
    <stocks:volume>1323580</stocks:volume>
    
  
  </stocks:stock>
  

</stocks:stocks>


### Unknown symbol → 404

In [7]:
resp = requests.get(f"{BASE_URL}/stocks/NOSUCHSYMBOL", timeout=30)
print(f"{resp.status_code} {resp.reason}")
assert resp.status_code == 404

404 Not Found


## 4. Historical stock lookup

`GET /stocks/{symbol}.{date}` looks up a stock on a specific trading date (`YYYY-MM-DD`).

The resolution order is:
1. **Local SQLite** — return the archived record if present.
2. **Proxy fallback** — otherwise fetch from `https://phisix-api2.appspot.com/stocks/{SYMBOL}.{date}.json`, return it, and **cache it locally** for next time.
3. If neither has data, return **404 Not Found**.

Both `.json` and `.xml` suffixes are supported on historical paths too.

In [8]:
DATE = "2023-09-03"  # YYYY-MM-DD

resp = requests.get(f"{BASE_URL}/stocks/{SYMBOL}.{DATE}", timeout=30)
show(resp)

GET http://localhost:8080/stocks/bpi.2023-09-03
-> 404 Not Found  |  content-type: 




In [9]:
# Historical lookup as XML
resp = requests.get(f"{BASE_URL}/stocks/{SYMBOL}.{DATE}.xml", timeout=30)
show(resp)

GET http://localhost:8080/stocks/bpi.2023-09-03.xml
-> 404 Not Found  |  content-type: 




## 5. Archive current feed

`GET` or `POST /stocks/archive` fetches the current live feed and saves/updates it in the local SQLite database. This is typically wired to a cron job so that daily snapshots accumulate for later historical queries.

Returns `200 OK` with the plain-text body `Archived successfully`.

In [10]:
# GET form
resp = requests.get(f"{BASE_URL}/stocks/archive", timeout=30)
print(f"GET  -> {resp.status_code} {resp.reason}: {resp.text!r}")

# POST form (equivalent)
resp = requests.post(f"{BASE_URL}/stocks/archive", timeout=30)
print(f"POST -> {resp.status_code} {resp.reason}: {resp.text!r}")

GET  -> 200 OK: 'Archived successfully'
POST -> 200 OK: 'Archived successfully'


### Confirm the archive worked

After archiving today's feed, a historical lookup for **today's date** should now resolve from the local database instead of the proxy.

In [11]:
import datetime
import zoneinfo

# The API uses Asia/Manila (GMT+8) for trading dates
today_manila = datetime.datetime.now(zoneinfo.ZoneInfo("Asia/Manila")).strftime("%Y-%m-%d")
print(f"Today (Asia/Manila): {today_manila}\n")

resp = requests.get(f"{BASE_URL}/stocks/{SYMBOL}.{today_manila}", timeout=30)
show(resp)

Today (Asia/Manila): 2026-08-06

GET http://localhost:8080/stocks/bpi.2026-08-06
-> 200 OK  |  content-type: application/json

{
  "as_of": "2026-08-06T00:00:00+08:00",
  "stocks": [
    {
      "name": "Bank of the Philippine Islands",
      "symbol": "BPI",
      "price": {
        "currency": "PHP",
        "amount": 103.4
      },
      "percent_change": 0.0,
      "volume": 1323580
    }
  ]
}


## 6. Bonus — load the feed into a DataFrame

The JSON structure maps cleanly onto a tabular view for quick analysis. Requires `pandas` (`pip install pandas`).

In [12]:
import pandas as pd

data = requests.get(f"{BASE_URL}/stocks", timeout=30).json()

df = pd.json_normalize(data["stocks"])
df = df.rename(columns={"price.currency": "currency", "price.amount": "amount"})

print(f"Feed as_of: {data['as_of']}  |  {len(df)} stocks\n")

# Top 10 gainers by percent change
df.sort_values("percent_change", ascending=False).head(10)

Feed as_of: 2026-08-06T00:00:00+08:00  |  389 stocks



,name,symbol,percent_change,volume,currency,amount
237,"Paxys, Inc.",PAX,22.22,28000,PHP,2.42
159,"Ionics, Inc.",ION,11.03,8556000,PHP,1.51
372,Xurpas Inc.,X,10.53,390000,PHP,0.21
245,Philippine Estates Corporation,PHES,10.14,8810000,PHP,0.38
343,Cirtek Holdings Philippines Corporation,TECH,10.00,7895000,PHP,1.10
85,"Dominion Holdings, Inc.",DHI,9.66,6329000,PHP,8.74
62,"Concreat Holdings Philippines, Inc.",CHP,9.57,18529000,PHP,1.03
28,Atlas Consolidated Mining and Development Corp...,AT,9.25,10307600,PHP,14.88
53,Concrete Aggregates Corp. ``A``,CA,9.24,23340,PHP,57.95
21,Anglo Philippine Holdings Corporation,APO,9.22,8777000,PHP,1.54


## 7. Watchlist — last week's prices for a list of assets

Set `ASSETS` below to a **comma-separated string** of PSE symbols. The next cell fetches each symbol's closing price for each of the last 7 days via `GET /stocks/{symbol}.{date}` and lays them out in a table (rows = symbols, columns = dates).

Prices resolve from the local SQLite archive when available, otherwise from the upstream proxy. Days with no data — weekends, holidays, or symbols the archive/proxy doesn't have — are left blank.

In [13]:
import datetime
import zoneinfo

import pandas as pd

# --- Your watchlist ------------------------------------------------------
# A comma-separated string of PSE symbols (case-insensitive, spaces ignored).
ASSETS = "JFC, MER, IMI, URC, AEV, RCR, SECB, RFM, MWIDE, ION, BHI"
# -------------------------------------------------------------------------

symbols = [s.strip().upper() for s in ASSETS.split(",") if s.strip()]

# Build the last 7 calendar days in Asia/Manila (oldest -> newest).
manila = zoneinfo.ZoneInfo("Asia/Manila")
today = datetime.datetime.now(manila).date()
dates = [today - datetime.timedelta(days=i) for i in range(6, -1, -1)]
date_strs = [d.strftime("%Y-%m-%d") for d in dates]


def fetch_close(symbol, date_str):
    """Return the closing price for a symbol on a date, or None if unavailable.

    Uses GET /stocks/{symbol}.{date}, which resolves from the local SQLite
    archive first and falls back to the upstream proxy (caching the result).
    """
    try:
        r = requests.get(f"{BASE_URL}/stocks/{symbol}.{date_str}", timeout=30)
    except requests.exceptions.RequestException:
        return None
    if r.status_code != 200:
        return None
    stocks = r.json().get("stocks", [])
    return stocks[0]["price"]["amount"] if stocks else None


records = {sym: {d: fetch_close(sym, d) for d in date_strs} for sym in symbols}

table = pd.DataFrame.from_dict(records, orient="index", columns=date_strs)
table.index.name = "symbol"

print(f"Closing prices (PHP) for the last 7 days — as of {today} (Asia/Manila)")
print("Blank cells mean no data (non-trading day, or symbol not available).\n")
table

Closing prices (PHP) for the last 7 days — as of 2026-08-06 (Asia/Manila)
Blank cells mean no data (non-trading day, or symbol not available).



,2026-07-31,2026-08-01,2026-08-02,2026-08-03,2026-08-04,2026-08-05,2026-08-06
symbol,,,,,,,
JFC,148.800,None,None,150.000,152.700,152.500,152.500
MER,480.000,None,None,487.000,485.000,482.000,482.000
IMI,6.350,None,None,6.360,6.300,6.580,6.580
URC,59.100,None,None,58.950,58.700,58.500,58.500
AEV,35.500,None,None,36.500,36.950,35.800,35.800
RCR,7.300,None,None,7.180,7.160,7.240,7.240
SECB,67.500,None,None,67.800,68.400,68.100,68.100
RFM,5.400,None,None,5.410,5.330,5.320,5.320
MWIDE,4.080,None,None,4.180,4.220,4.240,4.240


## 8. Dividends Calendar — Scrape from PSE EDGE

The API can scrape the full dividends calendar directly from the [PSE EDGE](https://edge.pse.com.ph) portal and store it in the local SQLite database.

| Method | Path | Description |
|--------|------|-------------|
| `GET` / `POST` | `/dividends/scrape` | Scrape all dividend records from PSE EDGE → DB |
| `GET` | `/dividends` | List all dividends (JSON). Filter: `?from=YYYY-MM-DD&to=YYYY-MM-DD` |
| `GET` | `/dividends.json` | Explicit JSON suffix |
| `GET` | `/dividends.xml` | XML format |
| `GET` | `/dividends/{symbol}` | Dividends for a specific security (`.json`/`.xml` suffix) |
| `POST` | `/dividends` | Create/update dividend record(s) |
| `DELETE` | `/dividends/{symbol}/{ex_date}` | Delete a specific record |

Each dividend record contains:

```jsonc
{
  "name": "Globe Telecom, Inc.",
  "symbol": "COMMON",         // security type
  "dividend_type": "Cash",
  "dividend_rate": "Php25 per common share",  // raw from PSE
  "ex_date": "2026-08-17",
  "record_date": "2026-08-18",
  "payment_date": "2026-09-03",
  "circular_number": "C05886-2026"
}
```

### Scrape dividends from PSE EDGE

`GET /dividends/scrape` fetches all pages from the PSE EDGE AJAX endpoint, parses the HTML table, saves every record to the local SQLite database, and returns the full calendar as JSON.

This may take a few seconds as it iterates through all pages (with a polite 300ms delay between requests).

In [14]:
resp = requests.get(f"{BASE_URL}/dividends/scrape", timeout=120)
data = resp.json()

print(f"{resp.status_code} {resp.reason}")
print(f"as_of:  {data['as_of']}")
print(f"total:  {data['total']} dividend records scraped & saved\n")

# Show the first 3 entries
print(json.dumps(data["dividends"][:3], indent=2))

200 OK
as_of:  2026-08-06T00:00:00+08:00
total:  522 dividend records scraped & saved

[
  {
    "name": "A Brown Company, Inc.",
    "symbol": "BRNPB Series B",
    "dividend_type": "Cash",
    "dividend_rate": "Php 2.0625 per share",
    "ex_date": "2027-02-08",
    "record_date": "2027-02-09",
    "payment_date": "2027-02-23",
    "circular_number": "C00595-2026"
  },
  {
    "name": "A Brown Company, Inc.",
    "symbol": "BRNPC Series C",
    "dividend_type": "Cash",
    "dividend_rate": "Php 2.1875 per share",
    "ex_date": "2027-02-08",
    "record_date": "2027-02-09",
    "payment_date": "2027-02-23",
    "circular_number": "C00592-2026"
  },
  {
    "name": "Asia United Bank Corporation",
    "symbol": "COMMON",
    "dividend_type": "Cash",
    "dividend_rate": "P0.50",
    "ex_date": "2026-12-01",
    "record_date": "2026-12-02",
    "payment_date": "2026-12-18",
    "circular_number": "C04854-2026"
  }
]


### List all dividends

`GET /dividends` returns all dividend records from the database (most recent first, up to 200 by default).

Use the `from` and `to` query params to filter by ex-date range.

In [15]:
# All dividends with ex-dates in August 2026
resp = requests.get(f"{BASE_URL}/dividends", params={"from": "2026-08-01", "to": "2026-08-31"}, timeout=30)
data = resp.json()

print(f"Dividends with ex-dates in Aug 2026: {data['total']}\n")
print(json.dumps(data["dividends"][:5], indent=2))

Dividends with ex-dates in Aug 2026: 26

[
  {
    "name": "Megawide Construction Corporation",
    "symbol": "MWP7A",
    "dividend_type": "Cash",
    "dividend_rate": "PhP1.828275",
    "ex_date": "2026-08-03",
    "record_date": "2026-08-04",
    "payment_date": "2026-08-19",
    "circular_number": "C05455-2026"
  },
  {
    "name": "Megawide Construction Corporation",
    "symbol": "MWP7B",
    "dividend_type": "Cash",
    "dividend_rate": "PhP1.925175",
    "ex_date": "2026-08-03",
    "record_date": "2026-08-04",
    "payment_date": "2026-08-19",
    "circular_number": "C05453-2026"
  },
  {
    "name": "A Brown Company, Inc.",
    "symbol": "BRNP Series A",
    "dividend_type": "Cash",
    "dividend_rate": "Php 1.75 per share",
    "ex_date": "2026-08-04",
    "record_date": "2026-08-05",
    "payment_date": "2026-09-01",
    "circular_number": "C00599-2026"
  },
  {
    "name": "A Brown Company, Inc.",
    "symbol": "BRNPB Series B",
    "dividend_type": "Cash",
    "dividend_r

### Dividends by security symbol

`GET /dividends/{symbol}` returns all dividend records for a specific security type. Supports `.json` and `.xml` suffixes.

In [16]:
resp = requests.get(f"{BASE_URL}/dividends/COMMON", timeout=30)
show(resp)

GET http://localhost:8080/dividends/COMMON
-> 200 OK  |  content-type: application/json

{
  "as_of": "2026-08-06T00:00:00+08:00",
  "total": 294,
  "dividends": [
    {
      "name": "Asia United Bank Corporation",
      "symbol": "COMMON",
      "dividend_type": "Cash",
      "dividend_rate": "P0.50",
      "ex_date": "2026-12-01",
      "record_date": "2026-12-02",
      "payment_date": "2026-12-18",
      "circular_number": "C04854-2026"
    },
    {
      "name": "The Philippine Stock Exchange, Inc.",
      "symbol": "COMMON",
      "dividend_type": "Cash",
      "dividend_rate": "Php5.50",
      "ex_date": "2026-09-30",
      "record_date": "2026-10-01",
      "payment_date": "2026-10-12",
      "circular_number": "C02816-2026"
    },
    {
      "name": "Philippine National Bank",
      "symbol": "COMMON",
      "dividend_type": "Cash",
      "dividend_rate": "P1.65",
      "ex_date": "2026-09-17",
      "record_date": "2026-09-18",
      "payment_date": "2026-10-01",
      "cir

### Dividends as XML

Append `.xml` to get the XML representation.

In [17]:
resp = requests.get(f"{BASE_URL}/dividends.xml", params={"from": "2026-08-01", "to": "2026-08-31"}, timeout=30)
print(f"{resp.status_code} {resp.reason}  |  content-type: {resp.headers.get('content-type')}\n")

# Print first 20 lines of the XML
pretty = xml.dom.minidom.parseString(resp.text).toprettyxml(indent="  ")
print("\n".join(pretty.splitlines()[:20]))
print("  ...")

200 OK  |  content-type: application/xml

<?xml version="1.0" ?>
<dividends as_of="2026-08-06T00:00:00+08:00" total="26">
  
  
  <dividend symbol="MWP7A">
    
    
    <name>Megawide Construction Corporation</name>
    
    
    <dividend_type>Cash</dividend_type>
    
    
    <dividend_rate>PhP1.828275</dividend_rate>
    
    
    <ex_date>2026-08-03</ex_date>
    
    
    <record_date>2026-08-04</record_date>
  ...


### Create a dividend record manually

`POST /dividends` accepts a single dividend object or an array. Useful for adding records manually when the PSE EDGE data is incomplete or for testing.

In [18]:
manual_dividend = {
    "name": "Test Company Inc.",
    "symbol": "TEST",
    "dividend_type": "Cash",
    "dividend_rate": "P1.00 per share",
    "ex_date": "2099-01-15",
    "record_date": "2099-01-16",
    "payment_date": "2099-02-01",
    "circular_number": "TEST-001"
}

resp = requests.post(f"{BASE_URL}/dividends", json=manual_dividend, timeout=30)
print(f"POST -> {resp.status_code} {resp.reason}: {resp.text!r}")

# Verify it was saved
resp = requests.get(f"{BASE_URL}/dividends/TEST", timeout=30)
show(resp)

# Clean up
resp = requests.delete(f"{BASE_URL}/dividends/TEST/2099-01-15", timeout=30)
print(f"\nDELETE -> {resp.status_code} {resp.reason}")

POST -> 201 Created: 'Saved 1 dividend record(s)'
GET http://localhost:8080/dividends/TEST
-> 200 OK  |  content-type: application/json

{
  "as_of": "2026-08-06T00:00:00+08:00",
  "total": 1,
  "dividends": [
    {
      "name": "Test Company Inc.",
      "symbol": "TEST",
      "dividend_type": "Cash",
      "dividend_rate": "P1.00 per share",
      "ex_date": "2099-01-15",
      "record_date": "2099-01-16",
      "payment_date": "2099-02-01",
      "circular_number": "TEST-001"
    }
  ]
}

DELETE -> 204 No Content


### Bonus — Dividends calendar as a DataFrame

Load the dividend feed into a pandas DataFrame for analysis. Requires `pandas`.

In [19]:
import pandas as pd

data = requests.get(f"{BASE_URL}/dividends", timeout=30).json()

df = pd.DataFrame(data["dividends"])
print(f"Total dividend records: {len(df)}\n")

# Upcoming dividends (ex_date >= today)
import datetime, zoneinfo
today_str = datetime.datetime.now(zoneinfo.ZoneInfo("Asia/Manila")).strftime("%Y-%m-%d")
upcoming = df[df["ex_date"] >= today_str].sort_values("ex_date")

print(f"Upcoming dividends (ex_date >= {today_str}): {len(upcoming)}\n")
upcoming[["name", "symbol", "dividend_type", "dividend_rate", "ex_date", "payment_date"]].head(15)

Total dividend records: 200

Upcoming dividends (ex_date >= 2026-08-06): 42



,name,symbol,dividend_type,dividend_rate,ex_date,payment_date
41,Ayala Corporation,ACPAR,Cash,Php39.741875,2026-08-12,2026-08-29
40,"MREIT, Inc.",COMMON,Cash,Php0.2630,2026-08-13,2026-08-28
38,Cityland Development Corporation,COMMON,Cash,Php 0.0326,2026-08-14,2026-09-11
37,ACEN CORPORATION,ACENB,Cash,Php20.00000,2026-08-14,2026-09-01
36,ACEN CORPORATION,ACENA,Cash,Php17.83250,2026-08-14,2026-09-01
39,"Puregold Price Club, Inc.",COMMON,Cash,P0.79/Share,2026-08-14,2026-09-09
35,"Globe Telecom, Inc.",GLOBB,Cash,Fixed annual rate of 6.7631% or Php67.631 per ...,2026-08-17,2026-09-02
34,"Globe Telecom, Inc.",GLOBA,Cash,Fixed annual rate of 6.1179% or Php61.179 per ...,2026-08-17,2026-09-02
33,"Globe Telecom, Inc.",COMMON,Cash,Php25 per common share,2026-08-17,2026-09-03
28,Rockwell Land Corporation,COMMON,Cash,Php 0.1547 per share to common shareholders,2026-08-18,2026-09-11


## 9. Reference / Signature Portfolios (DragonFi-style)

The API provides endpoints to store, manage, and analyze **signature portfolio reference models** (such as DragonFi D15, Dividend Harvest, and eTrader Signature Portfolios).

| Method | Path | Description |
|--------|------|-------------|
| `GET` | `/portfolios` | List all signature portfolio models |
| `GET` | `/portfolios/{id_or_slug}` | Fetch target holdings, target allocation percentages, and rationale notes |
| `GET` | `/portfolios/{id_or_slug}/analysis` | **Live market enrichment**: combines target weights with live market prices, % change, volume, and sector allocation breakdown |
| `POST` | `/portfolios` | Create a custom signature portfolio |
| `DELETE` | `/portfolios/{id_or_slug}` | Delete a signature portfolio |


### List all signature portfolios

`GET /portfolios` lists all reference model portfolios currently stored in the SQLite database.

In [20]:
resp = requests.get(f"{BASE_URL}/portfolios", timeout=30)
data = resp.json()

print(f"{resp.status_code} {resp.reason}")
print(f"Total reference portfolios: {data['total']}\n")
print(json.dumps(data["portfolios"], indent=2))

200 OK
Total reference portfolios: 6

[
  {
    "id": 1,
    "name": "DragonFi High Yield Dividend Portfolio",
    "slug": "dragonfi-high-yield",
    "description": "A signature reference portfolio focusing on top PSE dividend-paying REITs and blue-chip equities with high yield and cashflow stability.",
    "category": "Dividend Yield",
    "risk_level": "Moderate",
    "total_holdings": 6,
    "updated_at": "2026-08-06T00:00:00+08:00"
  },
  {
    "id": 2,
    "name": "PSEi Core Growth Portfolio",
    "slug": "psei-core-growth",
    "description": "Balanced reference allocation tracking premier Philippine conglomerates and market leaders for capital appreciation.",
    "category": "Balanced Growth",
    "risk_level": "Moderate",
    "total_holdings": 5,
    "updated_at": "2026-08-06T00:00:00+08:00"
  },
  {
    "id": 3,
    "name": "eTrader Managed Signature Portfolio",
    "slug": "etrader-signature",
    "description": "Actively managed portfolio by eTrader (16+ years in PH equities

### Get reference portfolio details

`GET /portfolios/{id_or_slug}` returns the full model allocation, target weight percentages, and notes for a portfolio.

In [21]:
SLUG = "dragonfi-d15"

resp = requests.get(f"{BASE_URL}/portfolios/{SLUG}", timeout=30)
show(resp)

GET http://localhost:8080/portfolios/dragonfi-d15
-> 200 OK  |  content-type: application/json

{
  "id": 6,
  "name": "DragonFi Dividend Benchmark (D15)",
  "slug": "dragonfi-d15",
  "description": "DragonFi's carefully selected list of 15 high-quality dividend-paying stocks with strong earnings, reliable payouts, and clear potential for long-term dividend growth.",
  "category": "Dividend Benchmark",
  "risk_level": "Moderate",
  "holdings": [
    {
      "symbol": "AP",
      "name": "ABOITIZ POWER CORPORATION",
      "target_weight_pct": 6.67,
      "sector": "Industrials",
      "notes": "Yield 3.14% (TTM)"
    },
    {
      "symbol": "AREIT",
      "name": "AREIT INC.",
      "target_weight_pct": 6.67,
      "sector": "Property",
      "notes": "Yield 6.53% (TTM)"
    },
    {
      "symbol": "BPI",
      "name": "BANK OF THE PHILIPPINE ISLANDS",
      "target_weight_pct": 6.67,
      "sector": "Financials",
      "notes": "Yield 4.70% (TTM)"
    },
    {
      "symbol": "CBC",


### Portfolio Analysis — Live Market Enrichment

`GET /portfolios/{id_or_slug}/analysis` enriches the portfolio's target weightings with live PSE market prices, daily percent changes, trading volume, and computes the exact sector allocation breakdown.

In [22]:
resp = requests.get(f"{BASE_URL}/portfolios/{SLUG}/analysis", timeout=30)
data = resp.json()

print(f"Portfolio: {data['portfolio']['name']}")
print(f"Category:  {data['portfolio']['category']} | Risk: {data['portfolio']['risk_level']}\n")

print("Sector Allocation Breakdown:")
for sector, weight in data["sector_breakdown"].items():
    print(f"  • {sector:20s}: {weight:6.2f}%")

Portfolio: DragonFi Dividend Benchmark (D15)
Category:  Dividend Benchmark | Risk: Moderate

Sector Allocation Breakdown:
  • Holding Firms       :   6.67%
  • Industrials         :  33.35%
  • Services            :  13.34%
  • Property            :  20.01%
  • Financials          :  20.01%
  • Mining & Oil        :   6.67%


### Portfolio Analysis in a DataFrame

Load the live enriched holdings analysis into a pandas DataFrame.

In [23]:
import pandas as pd

resp = requests.get(f"{BASE_URL}/portfolios/dragonfi-d15/analysis", timeout=30).json()
df = pd.DataFrame(resp["holdings_analysis"])

print(f"=== {resp['portfolio']['name']} Holdings ===\n")
df[["symbol", "name", "sector", "target_weight_pct", "live_price", "live_percent_change", "live_volume"]]

=== DragonFi Dividend Benchmark (D15) Holdings ===



,symbol,name,sector,target_weight_pct,live_price,live_percent_change,live_volume
0,AP,ABOITIZ POWER CORPORATION,Industrials,6.67,43.00,-1.15,706100
1,AREIT,AREIT INC.,Property,6.67,37.50,0.67,1075700
2,BPI,BANK OF THE PHILIPPINE ISLANDS,Financials,6.67,103.40,-0.48,1323580
3,CBC,CHINA BANKING CORPORATION,Financials,6.67,52.50,-1.13,318990
4,CREIT,CITICORE ENERGY REIT CORP.,Property,6.67,3.40,0.00,917000
5,KEEPR,THE KEEPERS HOLDINGS INC.,Industrials,6.67,1.89,-2.07,3280000
6,LTG,LT GROUP INC.,Holding Firms,6.67,14.82,0.95,1231900
7,MBT,METROPOLITAN BANK & TRUST COMPANY,Financials,6.67,66.65,0.68,1370660
8,MER,MANILA ELECTRIC COMPANY,Industrials,6.67,482.00,-0.62,284480
9,MWC,MANILA WATER COMPANY INC.,Industrials,6.67,34.20,-0.87,1321400


---

That covers every endpoint the PHISIX API exposes — including the **dividends calendar** scraped from PSE EDGE. See the accompanying **Bruno** collection under `bruno/` for the same requests in an API-client format.